In [0]:
df_final = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/gold/employees_gold")
df_emp = spark.read.parquet("/Volumes/hr_project_catalog/hr_schema/hr_project_volume/bronze/employees") 

display(df_final)
display(df_emp)

id,name,department,location,manager,salary,salary_band,bonus,experience_level,days_employed,rating
1,Alice,Engineering,New York,Sara,95000,High,9500,Mid,1874,4
2,Bob,Marketing,Chicago,Tom,72000,Mid,7200,Mid,2497,3
4,David,HR,Austin,Asha,61000,Low,6100,Junior,1573,4
5,Eve,Marketing,Chicago,Tom,79000,Mid,7900,Senior,2899,4
7,Grace,HR,Austin,Asha,58000,Low,5800,Junior,1173,3
8,Heidi,Finance,Dallas,Leo,85000,Mid,8500,Junior,848,4
9,Ivan,Engineering,New York,Sara,91000,High,9100,Junior,1005,5
10,Judy,Marketing,Chicago,Tom,67000,Low,6700,Junior,783,3


id,name,dept,salary,join_date,gender,age,is_active,bonus,rating
1,Alice,Engineering,95000,2021-03-15,F,29,true,9500,4
2,Bob,Marketing,72000,2019-07-01,M,35,true,7200,3
3,Carol,Engineering,88000,2020-11-20,F,31,false,8800,5
4,David,HR,61000,2022-01-10,M,27,true,6100,4
5,Eve,Marketing,79000,2018-05-25,F,38,true,7900,4
6,Frank,Engineering,102000,2017-09-30,M,42,false,10200,5
7,Grace,HR,58000,2023-02-14,F,24,true,5800,3


In [0]:
gold_path = "/Volumes/hr_project_catalog/hr_schema/hr_project_volume/gold/"

In [0]:
df_final.write.mode("overwrite").parquet(gold_path + "final_report")

In [0]:
df_emp.createOrReplaceTempView("emp_view")
result_3 = spark.sql("""
    SELECT dept, avg(salary) as avg_salary, count(name) as head_count, sum(bonus) 
    FROM emp_view 
    GROUP BY dept 
    ORDER BY avg_salary DESC
""")
display(result_3)
result_3.write.mode("overwrite").parquet(gold_path + "dept_report")

dept,avg_salary,head_count,sum(bonus)
Engineering,95000.0,3,28500
Marketing,75500.0,2,15100
HR,59500.0,2,11900


In [0]:
df_final_report = spark.read.parquet(gold_path + "final_report")
display(df_final_report.schema)
print(df_final_report.count())
display(df_final_report.limit(5))

StructType([StructField('id', LongType(), True), StructField('name', StringType(), True), StructField('department', StringType(), True), StructField('location', StringType(), True), StructField('manager', StringType(), True), StructField('salary', LongType(), True), StructField('salary_band', StringType(), True), StructField('bonus', LongType(), True), StructField('experience_level', StringType(), True), StructField('days_employed', IntegerType(), True), StructField('rating', LongType(), True)])

8


id,name,department,location,manager,salary,salary_band,bonus,experience_level,days_employed,rating
1,Alice,Engineering,New York,Sara,95000,High,9500,Mid,1874,4
2,Bob,Marketing,Chicago,Tom,72000,Mid,7200,Mid,2497,3
4,David,HR,Austin,Asha,61000,Low,6100,Junior,1573,4
5,Eve,Marketing,Chicago,Tom,79000,Mid,7900,Senior,2899,4
7,Grace,HR,Austin,Asha,58000,Low,5800,Junior,1173,3


In [0]:
df_final.write.mode("overwrite").partitionBy("department").parquet(gold_path + "partitioned_report")